# Stage 2–3 experiments on a Colab GPU

Runs the pre-registered experiments from `reports/experiment_plan.md` on a GPU and saves every run to Google Drive, so a disconnect loses at most one seed. Re-running a cell skips seeds that already finished.

**Before you start:** `Runtime → Change runtime type → GPU` (T4 is enough).

**Rules this notebook follows (do not edit them away):**
* Only configs with `"evaluate_test": true` touch the test set. Those are the pre-registered final runs.
* If `freeze_splits.py --check` fails, **stop**. The data split differs from the one all baselines used. Do not re-freeze.
* No raw text is written to the run folders: only ids, labels and probabilities.

Timings in this notebook are **estimates** until the first runs finish. The real numbers are recorded in each run's `metrics.json`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"

## 1. Google Drive (stores runs and the HF model cache so they survive disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/mhrisk'
os.makedirs(BASE, exist_ok=True)
os.environ['MHRISK_RUNS_DIR'] = f'{BASE}/runs'
os.environ['HF_HOME'] = f'{BASE}/hf_cache'
print(os.environ['MHRISK_RUNS_DIR'])

## 2. Get the code
If the repository is **private**, add a GitHub token as a Colab secret named `GH_TOKEN` (key icon in the left sidebar, read-only `contents` scope is enough) and set `PRIVATE = True`.

In [ ]:
BRANCH = 'ccr-0b88166e-2l8p6v'
PRIVATE = False
import os
url = 'https://github.com/faizdevx/Multilingual-Mental-Health-Risk-.git'
if PRIVATE:
    from google.colab import userdata
    url = url.replace('https://', f"https://{userdata.get('GH_TOKEN')}@")
if not os.path.exists('/content/repo'):
    !git clone -q -b {BRANCH} {url} /content/repo
%cd /content/repo
!git log --oneline -1

In [ ]:
!pip install -q -r requirements-colab.txt
import importlib.metadata as m
print({p: m.version(p) for p in ['numpy','pandas','scikit-learn','transformers']})

## 3. Data: download, rebuild and **verify** against the frozen split

In [ ]:
!python scripts/download_data.py
!python scripts/prepare_data.py > /dev/null
!python scripts/freeze_splits.py --check

If the cell above printed `matches frozen split` three times, continue. If not, stop and report the error. Do **not** re-freeze.

In [ ]:
!python -m pytest -q 2>&1 | tail -n 3

## 4. Experiments (run in this order)

| Order | Config | Seeds | Test? | Est. GPU time (T4, unmeasured) |
|---|---|---|---|---|
| 1 | `R2_muril_multitask_baseline` | 5 | **yes** (pre-registered #1) | ~1–1.5 h |
| 2 | `E1_1_risk_len256`, `E1_2_risk_headtail256` | 2 each | no | ~1 h |
| 3 | `E3_1_muril_single_{sentiment,emotion,risk}` | 5 each | **yes** (pre-registered #4) | ~1.5 h |
| 4 | `E2_1_hingroberta_sentiment`, `E2_2_xlmr_sentiment` | 2 each | no | ~0.5 h |
| 5 | `E1_3_lr_grid` (9 cells × 1 seed, up to 10 epochs) | 1 | no | ~3–4 h, split over sessions |

In [ ]:
!python scripts/run_experiment.py configs/R2_muril_multitask_baseline.json

In [ ]:
!python scripts/run_experiment.py configs/E1_1_risk_len256.json configs/E1_2_risk_headtail256.json

In [ ]:
!python scripts/run_experiment.py configs/E3_1_muril_single_sentiment.json configs/E3_1_muril_single_emotion.json configs/E3_1_muril_single_risk.json

In [ ]:
!python scripts/run_experiment.py configs/E2_1_hingroberta_sentiment.json configs/E2_2_xlmr_sentiment.json

In [ ]:
!python scripts/run_experiment.py configs/E1_3_lr_grid.json

## 5. Summaries and paired comparisons (validation unless the experiment was pre-registered for test)

In [ ]:
!python scripts/aggregate_runs.py --out {BASE}/results
# Examples: compare on VALIDATION (allowed for selection)
!python scripts/aggregate_runs.py --out {BASE}/results --compare R2_muril_multitask_baseline E1_2_risk_headtail256 --split val --task risk --metric pr_auc
!python scripts/aggregate_runs.py --out {BASE}/results --compare E3_1_muril_single_sentiment E2_1_hingroberta_sentiment --split val --task sentiment --metric macro_f1 --scope language:hi-Latn
# Pre-registered TEST comparison (multi-task vs single-task)
!python scripts/aggregate_runs.py --out {BASE}/results --compare R2_muril_multitask_baseline E3_1_muril_single_risk --split test --task risk --metric pr_auc

## 6. Package results to bring back to the repo (no checkpoints, no raw text)

In [ ]:
!cd {BASE} && zip -qr mhrisk_results.zip runs results -x '*/checkpoint/*' && ls -lh mhrisk_results.zip
from google.colab import files
files.download(f'{BASE}/mhrisk_results.zip')